# CS 3892 / 5892 — Session 20 · Lean 4 II: transition systems in Lean

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-11-05-lean-4-transition-systems.ipynb)

**Thursday, November 5, 2026.** A transition system in Lean is a structure with an `init` and a
`step`; `Reachable` is an inductive predicate, so a fact about every reachable state is proved by
**induction on how the state was reached**. That induction is why an **inductive invariant** holds.
Then session 9's add-2 machine, where `x ≠ 5` is **true but not inductive**, and the **stronger**
invariant that proves it; the running counter with a power switch, proved for **every cap `N`**; and
how to **build** a reachable state, to show the bound is tight.

The definitions (`TS`, `Reachable`, `InductiveInvariant`, `Invariant`, `inductive_invariant_holds`)
are the ones HW4's `Defs.lean` gives you, with the same names. Every Lean snippet on the slides is a
region of a file in `lean/S20/`, and every output and goal the slides show is asserted by a script
in `python/`. Lean v4.33.0, core library only. Colab has no Lean editor, so the scripts print the
goals; to see them as you type, open `lean/` in VS Code with the Lean 4 extension.

## Setup

Run these two once. The second installs Lean: give it a minute or two.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-11-05-lean-4-transition-systems"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
LD  = ROOT / "sessions" / SESSION / "lean"
FIG = ROOT / "sessions" / SESSION / "figures"
SMV = ROOT / "sessions" / SESSION / "smv"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- Lean v4.33.0, installed by HW4's checker: elan (the Lean version manager, pinned by
# SHA-256) into ~/.elan, then Lean through it -- about 575 MB, a minute or two the first time,
# and nothing to install after that. A Lean you already have is used as it is.
out = run(PYD / "lean_cli.py")
assert "version 4.33.0" in out

## 1. Reachable, and why an inductive invariant holds

`Reachable` has two constructors: an initial state, or one step from a reachable state. So
`inductive_invariant_holds` is an induction with two cases, and the script prints Lean's goal in
each.

In [ ]:
out = run(PYD / "01_reachable_and_induction.py")
assert "an inductive invariant holds in every reachable state" in out

## 2. The counter with a power switch: every step, every cap `N`

The state is a structure (`on`, `x`); a step is an inductive relation with one constructor per
transition. `cases` on the step gives one goal per transition. The proof never names a value of
`N`, so it covers every cap at once: what nuXmv, one cap at a time, cannot do.

In [ ]:
display(SVG(filename=str(FIG / "switch.svg")))
out = run(PYD / "02_switch_every_step.py")
assert "for every cap N" in out

## 3. True, and not inductive: session 9's add-2, in Lean

Start at 0, add 2 forever. `x ≠ 5` holds in every reachable state, but 3 satisfies it and steps
to 5. In Lean, "not inductive" is a theorem.

In [ ]:
display(SVG(filename=str(FIG / "add2.svg")))
out = run(PYD / "03_true_but_not_inductive.py")
assert "3 is never reached" in out

## 4. Strengthening: prove "x is even", then derive `x ≠ 5`

The stronger invariant admits fewer states, and that is what makes it inductive. HW4 Part 3 asks
for the same move on a different system.

In [ ]:
out = run(PYD / "04_strengthen.py")
assert "P holds in every reachable state" in out

## 5. Building reachable states: the bound is tight

`reach_k` builds `⟨true, k⟩` for every `k ≤ N`, one `Reachable.step` at a time, by induction on
`k`. `bound_tight` uses it as the witness of an `∃`.

In [ ]:
out = run(PYD / "05_build_reachable_states.py")
assert "it is tight" in out

## 6. The whole project builds, on the standard axioms

`lake build`, as VS Code runs it, then `#print axioms` for every theorem: no `sorry`, and nothing
classical except the proof (section 9) that the counter's third INVARSPEC is false.

In [ ]:
show(LD / "S20.lean")
out = run(PYD / "06_whole_library.py")
assert "Build completed successfully" in out and "Quot.sound" in out

## 7. Where `h` comes from, and which way `rw [h]` goes

`h` is a name someone chose for a hypothesis, not a keyword: the statement binds it, `intro` binds
it, `obtain` binds the pieces of one, or nobody does and Lean shows `a✝`. `rw [h]` with
`h : y = x + 7` turns `y` into `x + 7`; `rw [← h]` goes the other way. (In real Lean, `rw` then
tries `rfl`, which the Natural Number Game's `rw` does not.)

In [ ]:
out = run(PYD / "07_names_and_rw.py")
assert "rw [h] goes left to right" in out

## 8. From SMV to Lean, mechanically: `smv2lean`

`scripts/smv2lean` (from [ttj/leansmv](https://github.com/ttj/leansmv)) turns an SMV model into a
Lean transition system: each `case ... esac` becomes an if-then-else chain, a free input such as
`press` becomes `∃ press'`, and each `INVARSPEC` a theorem stub. First NuSMV's verdicts on the two
models, then the translation of each, checked against the committed Lean files.

In [ ]:
# --- lark, the parser library smv2lean needs (pure Python, small) ---------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lark==1.1.9"], check=True)
import lark
print("lark", lark.__version__)

In [ ]:
# --- NuSMV, for the SMV models in section 8 (same cell as sessions 9-17) -------
# NuSMV is free (LGPL) from nusmv.fbk.eu. 2.7.1 is tried first; it needs glibc >= 2.38
# and libedit, which an older Colab image may lack, so the static 2.6.0 build is the
# fallback. Same verdicts. (nuXmv, its successor, gives the same verdicts on this file.)
import re, shutil
SMV = ROOT / "sessions" / SESSION / "smv"

# Each archive is pinned by SHA-256 (taken Oct 30 2026; FBK publishes none). A download that does
# not match is refused and never unpacked or run.
BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz",
           "894dc71f0997683f28d4dc3512e3a6b608e02b9700e5e813017931df2d11883c"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",
           "11b12e42bc04566c4ec1f7cc205fa8704382ac4d69e052154cbb59c9b7e8e08c")]

def _fetch(url, sha, dest):
    """Download; refuse unless the SHA-256 matches the pin; unpack with the top folder stripped."""
    import hashlib, tarfile, tempfile, urllib.request
    with tempfile.TemporaryDirectory() as tmp:
        arc = pathlib.Path(tmp) / url.rsplit("/", 1)[1]
        with urllib.request.urlopen(url, timeout=120) as r, open(arc, "wb") as fh:
            shutil.copyfileobj(r, fh)
        got = hashlib.sha256(arc.read_bytes()).hexdigest()
        if got != sha:
            raise RuntimeError(f"REFUSING {url}: SHA-256 {got} is not the pinned {sha}")
        with tarfile.open(arc) as tar:
            for m in tar.getmembers():
                parts = pathlib.Path(m.name).parts[1:]
                if not parts or m.issym() or m.islnk() or ".." in parts:
                    continue
                m.name = str(pathlib.Path(*parts))
                tar.extract(m, dest)

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "counter.smv")], capture_output=True, text=True, timeout=60)
        return "-- invariant" in r.stdout       # counter.smv has INVARSPECs only
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, sha in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        _fetch(url, sha, dest)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def verdicts(path):
    """Run NuSMV on a file; print and return {formula: verdict}, plus the full output.
    NuSMV does not print verdicts in file order, so match them by formula."""
    out = subprocess.run([NUSMV, str(path)], capture_output=True, text=True).stdout
    res = {}
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            res[re.sub(r"\s+", " ", m.group(1)).strip()] = m.group(2)
    for k, v in res.items():
        print(f"{v:6} {k}")
    return res, out

In [ ]:
display(SVG(filename=str(FIG / "modes_efsm.svg")))
show(SMV / "counter.smv")
res, _ = verdicts(SMV / "counter.smv")
assert res == {"x <= count_max": "true", "(mode = off -> x = 0)": "true", "x < count_max": "false"}, res
display(SVG(filename=str(FIG / "mutex.svg")))
show(SMV / "mutex.smv")
res, _ = verdicts(SMV / "mutex.smv")
assert res == {"!(crit_a & crit_b)": "true"}, res

In [ ]:
out = run(PYD / "08_smv_to_lean.py")
assert "both files are exactly what smv2lean makes" in out

## 9. The two-mode counter: one goal per case

`x ≤ 10`: `split at hx` gives one goal per line of `next(x)`'s `case`. `mode = off → x = 0`:
`cases mode <;> cases press` gives one goal per combination, and two of them end in a
contradiction. `x < 10` is false, as NuSMV said: Lean proves the negation from NuSMV's own trace.

In [ ]:
out = run(PYD / "09_counter_case_split.py")
assert "one per (mode, press)" in out

## 10. Mutual exclusion: nine cases

`cases a <;> cases b` gives one goal for each pair (a, b) before the step; `simp` evaluates the
step's if-chains in each. Read what is left: a guard that fails, or a step that would need
`move = 1` and `move = 2` at once.

In [ ]:
out = run(PYD / "10_mutex_case_split.py")
assert "ruled out by the invariant" in out

## Try it yourself

Open the folder icon (left) → `cs3892-examples/sessions/cs3892-2026-11-05-lean-4-transition-systems/lean/S20/`,
double-click a file, edit, save (Ctrl+S), and re-run its cell. A runtime is deleted when it
disconnects: download anything you want to keep.

- In `Add2.lean`, change the step to `y = x + 3`. Which theorem breaks? Is `x ≠ 5` still true?
  Find a stronger invariant that proves whatever is true now.
- In `Switch.lean`, add a fifth transition, `reset : CtrStep N ⟨true, x⟩ ⟨true, 0⟩`. Which proof
  needs a new case, and what is its goal?
- In `Switch.lean`, try to prove `x ≤ N - 1` instead of `x ≤ N`. Where does it fail, and which
  theorem tells you it must?
- In `smv/mutex.smv`, delete `& b != critical` from a's guard, then regenerate and rebuild:
  `!python3 scripts/smv2lean/to_course.py sessions/cs3892-2026-11-05-lean-4-transition-systems/smv/mutex.smv S20 sessions/cs3892-2026-11-05-lean-4-transition-systems/lean/S20/Mutex.lean`,
  then re-run section 10. Which of the nine cases now fails, and what does
  NuSMV say about the same file?
- The insertion-sort proof from Tuesday, step by step: session 19's notebook, section 7.